# Validation of proxy outputs for historical period

In [1]:
%cd ~/git/future_hail_global

/home/561/tr2908/git/future_hail_global


In [43]:
import sys
from pathlib import Path

import itertools

sys.path.append(f'{Path("~/git/xarray_parcel/").expanduser()!s}')
sys.path.append(f'{Path("~/git/warming_levels").expanduser()!s}')

import cartopy.crs as ccrs
import dask
import matplotlib.pyplot as plt
import modules.fut_hail as fh
import numpy as np
import xarray
from dask.distributed import Client

## Setup

In [3]:
_ = dask.config.set(**{'array.slicing.split_large_chunks': False})
client = Client(n_workers=12, threads_per_worker=1)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 12
Total threads: 12,Total memory: 125.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35495,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41265,Total threads: 1
Dashboard: http://127.0.0.1:34395/status,Memory: 10.43 GiB
Nanny: tcp://127.0.0.1:44505,


In [44]:
# ruff: noqa: E712                                # Don't check rule E712 in Ruff.
plt.show()  # Start the plotting engine.
plt.rcParams['font.size'] = 16  # Font size for plots.
plt.rcParams['axes.formatter.useoffset'] = False  # Don't use offsets in plots.

lats = {
    'asia': slice(0, 55),  # Latitudes for selected world regions.
    'oceania': slice(-48, 0),
    'north_america': slice(12, 60),
    'south_america': slice(-57, 12),
    'europe': slice(25, 70),
    'africa': slice(-37, 25),
}

lons = {
    'asia': slice(55, 147),  # Longitudes for selected world regions.
    'oceania': slice(98, 180),
    'north_america': slice(-130, -50),
    'south_america': slice(-100, -20),
    'europe': slice(-12, 55),
    'africa': slice(-20, 55),
}

region_names = {
    'asia': 'Asia',  # Display names for selected world regions.
    'oceania': 'Oceania',
    'north_america': 'North America',
    'south_america': 'South America',
    'europe': 'Europe',
    'africa': 'Africa',
}

In [4]:
dat, landmask = fh.read_processed_data(apply_landmask=False)
era5 = fh.era5_climatology(landmask=None)

In [5]:
dat

<xarray.Dataset> Size: 55GB
Dimensions:                                  (model: 8, epoch: 3, season: 4,
                                              year_num: 20, lat: 180, lon: 360,
                                              proxy: 9, month: 12)
Coordinates:
  * season                                   (season) <U3 48B 'DJF' ... 'SON'
  * year_num                                 (year_num) int64 160B 1 2 ... 19 20
  * lat                                      (lat) float64 1kB -89.5 ... 89.5
  * lon                                      (lon) float64 3kB -179.5 ... 179.5
  * epoch                                    (epoch) <U10 120B '2C' ... 'hist...
  * model                                    (model) <U13 416B 'CMCC-CM2-SR5'...
  * proxy                                    (proxy) object 72B 'Raupach2023_...
  * month                                    (month) int64 96B 1 2 3 ... 11 12
Data variables: (12/31)
    seasonal_mean_mixed_100_cape             (model, epoch, year_num, season, lat, lon) float64 995MB dask.array<chunksize=(1, 1, 10, 2, 90, 180), meta=np.ndarray>
    seasonal_mean_mixed_100_cin              (model, epoch, year_num, season, lat, lon) float64 995MB dask.array<chunksize=(1, 1, 10, 2, 90, 180), meta=np.ndarray>
    seasonal_mean_mixed_100_lifted_index     (model, epoch, year_num, season, lat, lon) float64 995MB dask.array<chunksize=(1, 1, 10, 2, 90, 180), meta=np.ndarray>
    seasonal_mean_lapse_rate_700_500         (model, epoch, year_num, season, lat, lon) float64 995MB dask.array<chunksize=(1, 1, 10, 2, 90, 180), meta=np.ndarray>
    seasonal_mean_temp_500                   (model, epoch, year_num, season, lat, lon) float64 995MB dask.array<chunksize=(1, 1, 10, 2, 90, 180), meta=np.ndarray>
    seasonal_mean_melting_level              (model, epoch, year_num, season, lat, lon) float64 995MB dask.array<chunksize=(1, 1, 10, 2, 90, 180), meta=np.ndarray>
    ...                                       ...
    annual_extreme_temp_500                  (model, epoch, year_num, lat, lon) float64 249MB dask.array<chunksize=(1, 1, 20, 180, 360), meta=np.ndarray>
    annual_extreme_melting_level             (model, epoch, year_num, lat, lon) float64 249MB dask.array<chunksize=(1, 1, 20, 180, 360), meta=np.ndarray>
    annual_extreme_shear_magnitude           (model, epoch, year_num, lat, lon) float64 249MB dask.array<chunksize=(1, 1, 20, 180, 360), meta=np.ndarray>
    seasonal_hail_days                       (model, epoch, year_num, proxy, season, lat, lon) float64 9GB dask.array<chunksize=(1, 1, 10, 1, 2, 90, 180), meta=np.ndarray>
    annual_hail_days                         (model, epoch, year_num, proxy, lat, lon) float64 2GB dask.array<chunksize=(1, 1, 20, 1, 180, 360), meta=np.ndarray>
    monthly_hail_days                        (model, epoch, year_num, proxy, month, lat, lon) float64 27GB dask.array<chunksize=(1, 1, 10, 1, 6, 90, 180), meta=np.ndarray>
Attributes: (12/17)
    CMIP_grid_wind_v:                     gn
    CMIP_grid_wind_u:                     gn
    CMIP_grid_temperature:                gn
    CMIP_grid_specific_humidity:          gn
    CMIP_grid_surface_pressure:           gn
    CMIP_grid_surface_wind_v:             gn
    ...                                   ...
    CMIP_model_name:                      CMCC-CM2-SR5
    CMIP_note:                            Pressure derived from version used ...
    history:                              Regridded to 1 x 1degree grid using...
    NCO:                                  netCDF Operators version 5.0.5 (Hom...
    epoch_dates:                          2037-2056
    regrid_method:                        bilinear

In [23]:
proxies = ['Raupach2023_updated', 'Eccel2012', 'SHIP_0.1']

In [28]:
models = xarray.merge([dat.sel(epoch='historical', proxy=proxies).annual_hail_days.mean('year_num'),
                       era5.annual_hail_days.sel(proxy=proxies).expand_dims({'model': ['ERA5']})])

In [ ]:
for region in region_names:
    d = [
        models.min(['model', 'proxy']).annual_hail_days.sel(lat=lats[region], lon=lons[region]),
        models.mean(['model', 'proxy']).annual_hail_days.sel(lat=lats[region], lon=lons[region]),
        models.max(['model', 'proxy']).annual_hail_days.sel(lat=lats[region], lon=lons[region]),
    ]

    fh.plot_map(
        d,
        title=['Minimum', 'Mean', 'Maximum'],
        cbar_label='',
        ncols=3,
        figsize=(12, 5),
        wspace=0.1,
        share_axes=True,
        cbar_shrink=0.3,
        file=f'results/supplementary/proxy_climatology_{region}.pdf',
    )